# 3. Responsible AI Analysis: Able vs. Amputee for Prosthetic Design

## Research Goal for Prosthetic Design

This notebook uses **Responsible AI (RAI) tools** to understand **which features and patterns transfer from able-bodied to amputee physiology**—and crucially, **where and why translation fails**.

We build a binary classifier: **Can we distinguish able-bodied from amputee signals?**

The answer drives prosthetic design:
- **High distinguishability:** Signals are fundamentally different → prosthetic needs user-specific training.
- **Low distinguishability:** Signals are similar → pre-trained able-bodied models may transfer.

We use RAI components to explain *which EMG features* drive these differences:
- **Feature Importance:** Which muscles/frequencies matter most for each cohort?
- **Error Analysis:** Which actions confuse the classifier? Why?
- **Counterfactuals:** What EMG changes would make amputee signals more alike-bodied-like?
- **Causal Insights:** Are differences in EMG amplitude, frequency content, or timing?

**Non-clinical scope:** Offline analysis for design research, not a clinical diagnostic tool.

## 2. Dataset and Features

**Cohorts:**
- **Intact (Able-Bodied):** 22-channel glove kinematics from Notebook 1.
- **Amputee:** 12-channel EMG from forearm during intended movements (Notebook 2).

**Binary Classification Goal:**
- Target: `1 = Able-bodied (Intact)`, `0 = Amputee` (Disabled).
- Purpose: Measure signal transferability and identify which features are cohort-specific.

**Features for RAI Analysis:**
Classical EMG features (stateless, interpretable):
- **MAV (Mean Absolute Value):** Overall signal magnitude (muscle activation strength).
- **RMS (Root Mean Square):** Signal power (energy in contraction).
- **Waveform Length:** Complexity of EMG patterns (fine motor control).

Why classical features for RAI?  
RAI algorithms (SHAP, error analysis, counterfactual generation) work best with interpretable features. Classical EMG features map directly to muscle physiology; frequency-domain FFT bins do not.

## 3. Setup and Configuration

In [ ]:
# Imports and setup
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML

# Shared utilities
import sys
sys.path.insert(0, str(Path.cwd().parent))
from utils import (
    set_seed, find_repo_root, get_data_root,
    load_xlsx_recording, load_mat_recording,
    extract_emg_features,
    classification_summary, save_experiment_json,
)

# Responsible AI imports
try:
    from responsibleai import RAIInsights
    from raiwidgets import ExplanationDashboard, ErrorAnalysisDashboard, CausalInsightsDashboard
    from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
    print("✓ Responsible AI libraries loaded")
except ImportError as e:
    print(f"⚠️  RAI import warning: {e}")

# ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_fscore_support
)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All libraries imported")

## 4. Configuration

In [ ]:
SEED = 42
EXPERIMENT_ID = "EXP-010-rai-ablebody-vs-amputee"

set_seed(SEED)
repo_root = find_repo_root()
data_root = get_data_root(repo_root)
results_dir = repo_root / 'results' / 'metrics'
results_dir.mkdir(parents=True, exist_ok=True)

print(f"SEED: {SEED}")
print(f"Data root: {data_root}")
print(f"Results dir: {results_dir}")

## 5. Data Loading and Feature Extraction

Load both cohorts, extract classical EMG features (MAV, RMS, waveform length), and create a binary classification dataset.

In [ ]:
print("Loading Able-Bodied (Intact) data...")
intact_dir = data_root / 'Intact'
intact_paths = sorted(intact_dir.glob('*_A1_E*.xlsx'))
intact_records = []
for path in intact_paths:
    recording = load_xlsx_recording(path)
    # Use glove (22 channels) as proxy for intact physiology
    # Classical features on glove data
    windowed = extract_emg_features(
        recording, path, cohort='Intact',
        window_samples=200, stride_samples=50, min_purity=0.80
    )
    intact_records.append(windowed)
    print(f"  {path.name}: {windowed[0].shape[0]} windows")

print("\nLoading Amputee data...")
amputee_dir = data_root / 'Amputee'
amputee_paths = sorted(amputee_dir.glob('S*_E*.mat'))
amputee_records = []
for path in amputee_paths:
    recording = load_mat_recording(path)
    windowed = extract_emg_features(
        recording, path, cohort='Amputee',
        window_samples=200, stride_samples=50, min_purity=0.80
    )
    amputee_records.append(windowed)
    print(f"  {path.name}: {windowed[0].shape[0]} windows")

# Concatenate and create binary labels
X_intact = np.concatenate([r[0] for r in intact_records], axis=0)
target_intact = np.concatenate([r[1] for r in intact_records])
groups_intact = np.concatenate([r[2] for r in intact_records])

X_amputee = np.concatenate([r[0] for r in amputee_records], axis=0)
target_amputee = np.concatenate([r[1] for r in amputee_records])
groups_amputee = np.concatenate([r[2] for r in amputee_records])

print(f"\nIntact samples: {X_intact.shape[0]} × {X_intact.shape[1]} features")
print(f"Amputee samples: {X_amputee.shape[0]} × {X_amputee.shape[1]} features")

## 6. Create Binary Classification Dataset

Combine both cohorts and create labels: `1 = Able-bodied`, `0 = Amputee`.

In [ ]:
# Combine datasets
X = np.vstack([X_intact, X_amputee])
y = np.hstack([
    np.ones(X_intact.shape[0], dtype=int),  # Intact = 1
    np.zeros(X_amputee.shape[0], dtype=int)  # Amputee = 0
])

# Extract subjects for stratification
subjects_intact = np.asarray([g.split(':', 1)[0] for g in groups_intact])
subjects_amputee = np.asarray([g.split(':', 1)[0] for g in groups_amputee])
subjects = np.hstack([subjects_intact, subjects_amputee])

print(f"Total samples: {X.shape[0]}")
print(f"Feature dimension: {X.shape[1]}")
print(f"Class distribution:")
print(f"  Able-bodied (1): {(y == 1).sum()}")
print(f"  Amputee (0): {(y == 0).sum()}")
print(f"\nUnique subjects: {len(np.unique(subjects))}")

## 7. Train/Test Split (Subject-Stratified)

Hold out subjects to prevent leakage and ensure fair generalization testing.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Stratified group split: hold out subjects, stratify by cohort
split_gen = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(split_gen.split(X, y, subjects))

# Use first fold for train/test
train_idx, test_idx = splits[0]

X_train = X[train_idx]
y_train = y[train_idx]
subjects_train = subjects[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]
subjects_test = subjects[test_idx]

print(f"Train set: {X_train.shape[0]} samples")
print(f"  Able-bodied: {(y_train == 1).sum()}, Amputee: {(y_train == 0).sum()}")
print(f"  Unique subjects: {len(np.unique(subjects_train))}")

print(f"\nTest set: {X_test.shape[0]} samples")
print(f"  Able-bodied: {(y_test == 1).sum()}, Amputee: {(y_test == 0).sum()}")
print(f"  Unique subjects: {len(np.unique(subjects_test))}")

## 8. Feature Scaling and Model Training

Standardize features and train a logistic regression classifier for interpretability with RAI tools.

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression (interpretable model for RAI)
print("Training binary classifier (Able-Bodied vs. Amputee)...")
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)
y_pred_proba_test = model.predict_proba(X_test_scaled)[:, 1]

print("✓ Model trained")

## 9. Model Evaluation

Assess cross-subject discriminability and fairness across cohorts.

# Test performance
test_acc = (y_pred_test == y_test).mean()
test_auc = roc_auc_score(y_test, y_pred_proba_test)

print(f"\nCross-Subject Performance:")
print(f"  Accuracy: {test_acc:.1%}")
print(f"  ROC-AUC: {test_auc:.1%}")

print(f"\nClassification Report (Test Set):")
print(classification_report(
    y_test, y_pred_test,
    target_names=['Amputee', 'Able-Bodied'],
    digits=3
))

# Fairness analysis
from sklearn.metrics import balanced_accuracy_score
balanced_acc = balanced_accuracy_score(y_test, y_pred_test)
print(f"\nFairness Metrics:")
print(f"  Balanced Accuracy: {balanced_acc:.1%}")
print(f"  (Equal weight to both cohorts)")

## 10. Responsible AI: Feature Importance

Which EMG features drive the ability-to-amputee distinction? Using SHAP for model-agnostic explanation.

In [ ]:
# SHAP-based feature importance
try:
    import shap
    
    # Use a sample for speed
    sample_size = min(200, X_test_scaled.shape[0])
    sample_idx = np.random.choice(X_test_scaled.shape[0], sample_size, replace=False)
    
    explainer = shap.LinearExplainer(model, X_train_scaled[:500])  # Use training background
    shap_values = explainer.shap_values(X_test_scaled[sample_idx])
    
    # Mean absolute SHAP for global importance
    feature_importance = np.abs(shap_values).mean(axis=0)
    
    # Create interpretable feature names
    feature_names = [f"Channel {i//3} - {'MAV' if i % 3 == 0 else ('RMS' if i % 3 == 1 else 'WL')}" 
                     for i in range(X.shape[1])]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    top_idx = np.argsort(feature_importance)[-15:]
    ax.barh(range(len(top_idx)), feature_importance[top_idx])
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([feature_names[i] for i in top_idx])
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('Feature Importance for Able-Bodied vs. Amputee Classification')
    plt.tight_layout()
    plt.savefig(results_dir / f"{EXPERIMENT_ID}_shap_importance.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Feature importance computed")
    
except ImportError:
    print("⚠️  SHAP not available. Skipping SHAP analysis.")
    feature_importance = np.abs(model.coef_[0])
    print(f"Using model coefficients for importance instead.")

## 11. Error Analysis: Where Does the Classifier Fail?

Identify confusions between cohorts and associated actions.

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

print(f"\n### ERROR ANALYSIS ###\n")
print(f"Confusion Matrix:")
print(f"  True Negatives (Amputee correctly classified):  {tn}")
print(f"  False Positives (Amputee → Able-Bodied):        {fp}")
print(f"  False Negatives (Able-Bodied → Amputee):        {fn}")
print(f"  True Positives (Able-Bodied correctly classified): {tp}")

amputee_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
ablebody_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\nPer-Cohort Recall:")
print(f"  Amputee recall: {amputee_recall:.1%} (how many amputee samples correctly identified)")
print(f"  Able-Bodied recall: {ablebody_recall:.1%} (how many able-bodied samples correctly identified)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Amputee', 'Able-Bodied'],
            yticklabels=['Amputee', 'Able-Bodied'])
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix: Able-Bodied vs. Amputee Classification')
plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_confusion.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n### DESIGN IMPLICATION ###")
print(f"False positives ({fp}): Amputee samples misclassified as Able-Bodied")
print(f"  → These amputee EMG patterns look like intact kinematics.")
print(f"  → May benefit from pre-trained able-bodied models.")

print(f"\nFalse negatives ({fn}): Able-Bodied samples misclassified as Amputee")
print(f"  → These able-bodied patterns look like amputee EMG.")
print(f"  → Suggests some kinematics are intrinsically amputee-like.")

## 12. Decision Boundaries and Calibration

Examine prediction confidence and the overlap region where cohorts are most similar.

In [ ]:
# Prediction confidence
pred_proba_amputee = model.predict_proba(X_test_scaled[y_test == 0])[:, 1]
pred_proba_ablebody = model.predict_proba(X_test_scaled[y_test == 1])[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograms
axes[0].hist(pred_proba_amputee, bins=20, alpha=0.6, label='Amputee', color='red')
axes[0].hist(pred_proba_ablebody, bins=20, alpha=0.6, label='Able-Bodied', color='blue')
axes[0].axvline(0.5, color='black', linestyle='--', label='Decision Boundary')
axes[0].set_xlabel('Model Confidence (P(Able-Bodied))')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].legend()

# ROC Curve
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate (Amputee → Able-Bodied)')
axes[1].set_ylabel('True Positive Rate (Able-Bodied correctly identified)')
axes[1].set_title('ROC Curve: Separability of Cohorts')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_confidence_calibration.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n### TRANSFERABILITY INSIGHTS ###\n")
if roc_auc > 0.90:
    print(f"✓ High separability (AUC = {roc_auc:.2f}):")
    print(f"  → Able-bodied and amputee signals are very different.")
    print(f"  → Cross-cohort transfer unlikely; prosthetic needs user-specific training.")
elif roc_auc > 0.70:
    print(f"⚠️  Moderate separability (AUC = {roc_auc:.2f}):")
    print(f"  → Some patterns transfer, others are cohort-specific.")
    print(f"  → Consider hybrid: pre-trained model + short user calibration.")
else:
    print(f"✓ Low separability (AUC = {roc_auc:.2f}):")
    print(f"  → Signals largely overlap; strong cross-cohort transfer potential.")
    print(f"  → Pre-trained able-bodied models may work for amputees.")

## 13. Actionable Design Recommendations

Synthesize RAI findings into prosthetic design constraints and mitigation strategies.

In [ ]:
# Design recommendation table
recommendations = pd.DataFrame([
    {
        'Finding': 'Model Accuracy',
        'Value': f'{test_acc:.1%}',
        'Implication': 'How well we can distinguish able-bodied from amputee signals.',
        'Prosthetic Design Constraint': 
            'High accuracy → signals are cohort-specific. Low accuracy → signals overlap (transfer-friendly).'
    },
    {
        'Finding': 'ROC-AUC',
        'Value': f'{test_auc:.1%}',
        'Implication': 'Probability that the model ranks able-bodied higher than amputee.',
        'Prosthetic Design Constraint': 
            'AUC > 0.90: prepare for user-specific tuning. AUC < 0.70: pre-trained models may suffice.'
    },
    {
        'Finding': f'Amputee Recall ({amputee_recall:.1%})',
        'Value': f'{amputee_recall:.1%}',
        'Implication': 'Fraction of amputee samples correctly identified as distinct.',
        'Prosthetic Design Constraint': 
            'Low recall → many amputee samples resemble able-bodied. May reduce false activations in prosthetic.'
    },
    {
        'Finding': f'Able-Bodied Recall ({ablebody_recall:.1%})',
        'Value': f'{ablebody_recall:.1%}',
        'Implication': 'Fraction of able-bodied samples correctly identified.',
        'Prosthetic Design Constraint': 
            'High recall → able-bodied patterns are distinct. Useful for transfer learning baseline.'
    },
])

print("\n### DESIGN DECISION TABLE ###\n")
for idx, row in recommendations.iterrows():
    print(f"\n{idx + 1}. {row['Finding']}: {row['Value']}")
    print(f"   Implication: {row['Implication']}")
    print(f"   → Constraint: {row['Prosthetic Design Constraint']}")

# Save table
recommendations.to_csv(results_dir / f"{EXPERIMENT_ID}_design_recommendations.csv", index=False)
print(f"\n✓ Design recommendations saved")

## 14. Summary and Next Steps

Synthesize findings for presentation and future work.

In [ ]:
print(f"""
\n### SUMMARY: ABLE-BODIED TO AMPUTEE TRANSLATION ###

**Notebook 1 (Able-Bodied Baseline):**
  - Established performance ceiling for action recognition from glove kinematics.
  - Identified which actions are intrinsically distinguishable (F1 > 0.75).

**Notebook 2 (Amputee Reality):**
  - Measured EMG-based action recognition in residual limbs.
  - Quantified translation gap (how much performance drops compared to Notebook 1).
  - Identified which actions transfer well and which require special handling.

**Notebook 3 (Responsible AI - This Notebook):**
  - Binary classification: Can we distinguish able-bodied from amputee signals?
  - Feature importance: Which EMG channels/muscles drive the distinction?
  - Confusion patterns: Where do cohorts overlap? Where are they most different?
  - Design recommendations:
    ✓ High ROC-AUC → Pre-train on able-bodied, fine-tune per user.
    ✓ Low ROC-AUC → Pre-trained able-bodied models may transfer directly.
    ✓ High false positives → Certain amputee patterns are able-bodied-like (leverage in transfer).
    ✓ High false negatives → Certain able-bodied patterns are amputee-like (prepare user for mismatch).

**Prosthetic Design Implications:**
  1. If high separability (AUC > 0.90): Prosthetic needs user-specific calibration.
  2. If moderate separability (0.70 < AUC < 0.90): Hybrid approach (pre-train + fine-tune).
  3. If low separability (AUC < 0.70): Pre-trained able-bodied models likely effective.
  4. Focus on high-confidence actions from Notebook 2; deprioritize low-F1 actions or add user confirmation.

**Next Steps for Research:**
  - Develop user-specific calibration workflows based on ROC curve insights.
  - Prioritize features identified in importance analysis for prosthetic sensor design.
  - Test counterfactual scenarios: which EMG features must change for amputee→able-bodied transfer?
  - Validate with external amputee cohorts and real prosthetic devices.
""")

## 15. Save Results

In [ ]:
# Save comprehensive experiment summary
experiment_data = {
    'experiment_id': EXPERIMENT_ID,
    'analysis_type': 'Responsible AI - Binary Cohort Classification',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'test_subjects': sorted(np.unique(subjects_test).tolist()),
    'metrics': {
        'accuracy': float(test_acc),
        'roc_auc': float(test_auc),
        'balanced_accuracy': float(balanced_acc),
        'amputee_recall': float(amputee_recall),
        'ablebody_recall': float(ablebody_recall),
    },
    'confusion_matrix': {
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
    },
    'data': {
        'n_samples_train': int(X_train.shape[0]),
        'n_samples_test': int(X_test.shape[0]),
        'n_features': int(X.shape[1]),
        'ablebody_samples': int((y == 1).sum()),
        'amputee_samples': int((y == 0).sum()),
    },
    'model': 'LogisticRegression',
    'rai_components': ['Feature Importance (SHAP)', 'Error Analysis', 'Calibration Analysis'],
    'design_recommendations': recommendations.to_dict(orient='records'),
}

save_experiment_json(results_dir, EXPERIMENT_ID, experiment_data)
print(f"✓ RAI experiment results saved: {EXPERIMENT_ID}")
print(f"\n✓ All three notebooks complete!")
print(f"  - Notebook 1: Able-bodied baseline (frequency domain action recognition)")
print(f"  - Notebook 2: Amputee reality (translation gap measurement)")
print(f"  - Notebook 3: Responsible AI (transfer analysis for prosthetic design)")